# 🌟 Visual Vox: 视障全语音多模态无障碍视觉交互系统

> **项目名称**：Visual Vox (Multimodal Voice-Guided Visual Accessibility Assistant)  
> **核心理念**：传统无障碍视觉辅助产品通常只能输出一句简短的图像概述（如“画面中有一只猫”），视障人士无法感知更微观、生动的视觉层次。  
> **Visual Vox 创新机制**：利用大模型的多模态视觉认知与语义拆解能力，建立**“分层递归树状感知（Hierarchical Recursive Zoom-in）”**模型。视障人士不仅能“听懂”整幅图，更能通过**自然语音对话（STT）**与**高保真语音播报（TTS）**，随心所欲地“深入聚焦”任何感兴趣的物体与微观部件！

### 📐 系统工作流架构

```
  [ 输入图像 (URL / 本地文件) ]
             │
             ▼ (多模态视觉解析 GPT-4o)
    【第 1 层：全局印象与宏观对象】
     ├── 语音播报整体印象 (TTS: Tingting / OpenAI Nova)
     └── 列出画面中的对象：[1] 橘色小猫  [2] 浅蓝绒毯
             │
             ▼ 视障用户语音提问: "看看这只猫的细节" (STT Whisper / 键盘)
    【第 2 层：对象微观拆解】
     ├── 播报小猫姿态细节
     └── 识别子部件：[1] 眼睛  [2] 鼻子  [3] 猫爪  [4] 耳朵
             │
             ▼ 视障用户语音提问: "我想摸摸它的爪子，爪子长什么样？"
    【第 3 层：微观特写感知】
     ├── 播报爪垫触感细节 (肉垫、掌球、收拢爪尖)
     └── 到达叶子节点（递归自然终止）
```

In [ ]:
# 1. 基础依赖检查与安装
# 提示：若本地未配置 API Key，系统会自动无缝切换至内置仿真演示模式！
!pip install -q openai

In [ ]:
import os
import sys

# 导入 Visual Vox 模块
sys.path.append('..')
from visual_vox import VisualVoxAgent

print("✅ Visual Vox 引擎与依赖加载完成！")

### 🚀 运行交互式体验

支持**自然语言语音指令**（如输入“看看猫爪”、“眼睛”、“返回上一层”）与**数字按键（1/2/3/b/q）**双模交互！

In [ ]:
# 初始化 Agent (支持传入 api_key="sk-..."，不传则自动进入仿真模式)
agent = VisualVoxAgent(api_key=None, enable_tts=True)

# 示例测试图片（猫咪图）
test_image = "https://images.unsplash.com/photo-1543852786-1cf6624b9987?auto=format&fit=crop&w=800&q=80"

print("=" * 65)
print("           Visual Vox 全语音多模态无障碍会话启动")
print("=" * 65)

agent.say("正在为您分析图片，请稍候。")
state = agent.analyze_image(test_image)

while True:
    print("\n" + "=" * 50)
    print(f"【当前层级感知】: {state.get('overall')}")
    print("=" * 50)
    agent.say(state.get('overall'))

    objects = state.get("objects", {})
    obj_keys = list(objects.keys())

    if not obj_keys:
        msg = "该部位已是最终微观细节，无法继续深入拆解。输入 b 可返回上一层。"
        print(msg)
        agent.say(msg)
    else:
        print("\n可供探查的部件列表:")
        for idx, k in enumerate(obj_keys, 1):
            print(f"  [{idx}] {k} -> {objects[k]}")
        menu_speech = "当前可观察：" + "，".join(obj_keys) + "。请输入编号或说出想看的部位。"
        agent.say(menu_speech)

    user_input = input("\n[用户指令] 请输入编号或部位名称（'b' 返回上一级，'q' 退出）: ").strip()

    if not user_input or user_input.lower() == 'q':
        agent.say("感谢使用 Visual Vox，再见！")
        print("会话已退出。")
        break

    if user_input.lower() == 'b':
        prev = agent.go_back()
        if prev:
            agent.say("已返回上一层。")
            state = prev
        else:
            agent.say("已经是顶层全局画面，无法再返回。")
        continue

    # 1. 数字编号解析
    chosen_target = None
    if user_input.isdigit() and 1 <= int(user_input) <= len(obj_keys):
        chosen_target = obj_keys[int(user_input) - 1]
    else:
        # 2. 语义模糊匹配
        chosen_target = agent.recognizer.parse_target(user_input, obj_keys)

    if chosen_target:
        agent.say(f"正在为您深入探查：【{chosen_target}】。")
        agent.drill_down(chosen_target)
        state = agent.current_state
    else:
        err_msg = f"未能在当前列表中识别到与‘{user_input}’相关的部件，请重新选择。"
        print(err_msg)
        agent.say(err_msg)